### Import Acceptance data

In [7]:
import pandas as pd
data_dir = "C:\\Users\\AM000098\\Desktop\\Lenovo\\ITE\\AllSim\\src\\myallsim\\data\\my_destination"
data_dir = "my_acceptance_data"
data = pd.read_csv(f"{data_dir}/acceptance_data_synth.csv")
train = pd.read_csv(f"{data_dir}/acceptance_data_synth_train.csv")
test = pd.read_csv(f"{data_dir}/acceptance_data_synth_test.csv")

In [8]:
print(train.shape)
print(test.shape)

(17997, 75)
(948, 75)


In [9]:
data["ACC"].value_counts() 

ACC
0    18185
1      760
Name: count, dtype: int64

In [10]:
train["ACC"].value_counts() 

ACC
0    17267
1      730
Name: count, dtype: int64

In [12]:
train = train.drop(["Unnamed: 0"], axis = 1)
test = test.drop(["Unnamed: 0"], axis = 1)

## Train acc+dt estimator

In [46]:
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Activation, Lambda, LeakyReLU, PReLU
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import roc_auc_score

In [60]:
X_test = test.drop(columns=["ACC", "DEC_TIME"])
y_test = test[["ACC", "DEC_TIME"]]

train, val = train_test_split(train, test_size=0.05)

X_train = train.drop(columns=["ACC", "DEC_TIME"])
y_train = train[["ACC", "DEC_TIME"]]

X_val = val.drop(columns=["ACC", "DEC_TIME"])
y_val = val[["ACC", "DEC_TIME"]]

In [99]:
model = Sequential()

model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(2, activation='linear'))

model.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_109 (Dense)                    │ (None, 64)                  │           4,672 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_110 (Dense)                    │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_111 (Dense)                    │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,962 (35.01 KB)

 Trainable params: 8,962 (35.01 KB)

 Non-trainable params: 0 (0.00 B)

In [100]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

Epoch 1/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 95.4514 - val_loss: 0.8477
Epoch 2/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0444 - val_loss: 0.4763
Epoch 3/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4166 - val_loss: 0.3357
Epoch 4/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3304 - val_loss: 0.5699
Epoch 5/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.9727 - val_loss: 0.2251
Epoch 6/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3341 - val_loss: 0.1935
Epoch 7/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3915 - val_loss: 0.2226
Epoch 8/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.5082 - val_loss: 0.5680
Epoch 9/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3033 - val_loss: 0.1659
Epoch 10/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1785 - val_loss: 0.2576
Epoch 11/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1881 - val_loss: 0.4017
Epoch 12/20
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/ste

In [101]:
# Instantiate the loss functions
binary_crossentropy = tf.keras.losses.BinaryCrossentropy()
mean_squared_error = tf.keras.losses.MeanSquaredError()

@tf.function  # This ensures TensorFlow works properly in graph mode
def custom_loss(y_true, y_pred):
    # y_true[:, 0] is the true values for ACC, y_pred[:, 0] is the predicted values for ACC
    acc_loss = binary_crossentropy(y_true[:, 0], y_pred[:, 0])
    
    # y_true[:, 1] is the true values for DEC_TIME, y_pred[:, 1] is the predicted values for DEC_TIME
    dec_time_loss = mean_squared_error(y_true[:, 1], y_pred[:, 1])
    
    # Combine the two losses (you can weight them differently if necessary)
    total_loss = acc_loss + dec_time_loss
    
    return total_loss

In [166]:
# Define the input layer
from tensorflow.keras.regularizers import l2
inputs = Input(shape=(X_train.shape[1],))

# Hidden layers
x = Dense(64, kernel_regularizer=l2(0.01))(inputs)
x = LeakyReLU(alpha=0.01)(x)
x = Dense(32, kernel_regularizer=l2(0.01))(x) 
x = LeakyReLU(alpha=0.01)(x)

# Separate paths for ACC and DEC_TIME
# x_acc = Dense(16, activation='relu')(x)
#x_acc = Dense(32, activation='relu')(x_acc)
acc_output = Dense(1, activation='sigmoid', name='acc_output')(x)

#x_dec = Dense(16, activation='relu')(x)
#x_dec = Dense(32, activation='relu')(x_dec)
dec_time_output = Dense(1, activation='relu')(x)
#dec_time_output = Lambda(lambda x: tf.clip_by_value(x, 0.0, 2.0), name='dec_time_output')(dec_time_output)  # Clip to [0, 2]

# Combine the outputs
outputs = tf.keras.layers.concatenate([acc_output, dec_time_output])

# Define the model
NNmodel_relu = Model(inputs=inputs, outputs=outputs)

# Compile the model
NNmodel_relu.compile(optimizer='adam', loss=custom_loss)

# Summary of the model
NNmodel_relu.summary()

C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)    │ (None, 72)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_30 (Dense)              │ (None, 64)                │           4,672 │ input_layer_8[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_20 (LeakyReLU)    │ (None, 64)                │               0 │ dense_30[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_31 (Dense)              │ (None, 32)                │           2,080 │ leaky_re_lu_20[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_21 (LeakyReLU)    │ (None, 32)                │               0 │ dense_31[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ acc_output (Dense)            │ (None, 1)                 │              33 │ leaky_re_lu_21[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_32 (Dense)              │ (None, 1)                 │              33 │ leaky_re_lu_21[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_7 (Concatenate)   │ (None, 2)                 │               0 │ acc_output[0][0],          │
│                               │                           │                 │ dense_32[0][0]             │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 6,818 (26.63 KB)

 Trainable params: 6,818 (26.63 KB)

 Non-trainable params: 0 (0.00 B)

In [165]:
# Assuming 'ACC' is the column name for the first target (binary classification)
y_acc = y_train['ACC'].values  # Convert ACC to NumPy array


# Compute class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_acc),
    y=y_acc
)

# Create sample weights for each sample in y_train['ACC']
sample_weights = np.where(y_acc == 0, class_weights[0], class_weights[1])
sample_weights = np.where(y_acc == 0, 1/(len(y_acc) - np.count_nonzero(y_acc)), 1/np.count_nonzero(y_acc))

# Check shape of sample_weights, should match number of samples
print(class_weights)
print(sample_weights)
print(y_train)

[ 0.521446   12.15718563]
[6.42095801e-05 6.42095801e-05 6.42095801e-05 ... 6.42095801e-05
 6.42095801e-05 6.42095801e-05]
       ACC  DEC_TIME
2716     0  0.392689
7376     0  0.153994
10175    0  0.397203
16773    0  0.226281
15033    0  0.426607
...    ...       ...
3092     0  0.150349
176      0  0.416375
15803    0  0.241724
2886     0  0.208464
14040    0  0.265431

[16242 rows x 2 columns]


In [167]:
early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

history = NNmodel_relu.fit(
    X_train, y_train,
    #sample_weight=sample_weights,
    epochs=1000,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

Epoch 1/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 1.2989 - val_loss: 0.8948
Epoch 2/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8432 - val_loss: 0.7252
Epoch 3/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7814 - val_loss: 0.6911
Epoch 4/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7473 - val_loss: 0.7220
Epoch 5/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7310 - val_loss: 0.6411
Epoch 6/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6934 - val_loss: 0.6615
Epoch 7/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6678 - val_loss: 0.6038
Epoch 8/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6492 - val_loss: 0.6225
Epoch 9/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6715 - val_loss: 0.5761
Epoch 10/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6131 - val_loss: 0.5642
Epoch 11/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5961 - val_loss: 0.5545
Epoch 12/1000
508/508 ━━━━━━━━

In [154]:
early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

history = NNmodel_relu.fit(
    X_train, y_train,
    #sample_weight=sample_weights,
    epochs=1000,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

Epoch 1/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 655.0576 - val_loss: 2.9199
Epoch 2/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2.6333 - val_loss: 1.9229
Epoch 3/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.9161 - val_loss: 1.4772
Epoch 4/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.5468 - val_loss: 1.2973
Epoch 5/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3281 - val_loss: 1.1352
Epoch 6/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.2123 - val_loss: 1.0570
Epoch 7/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.1352 - val_loss: 1.0406
Epoch 8/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0347 - val_loss: 0.9175
Epoch 9/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9814 - val_loss: 0.9136
Epoch 10/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.9609 - val_loss: 0.7987
Epoch 11/1000
508/508 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.0371 - val_loss: 0.8078
Epoch 12/1000
508/508 ━━━━━━

In [54]:
# TEST
test_loss = NNmodel_relu.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1164
Test Loss: 0.11927667260169983


In [129]:
# TEST
test_loss = NNmodel_relu.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1079 
Test Loss: 0.11864766478538513


In [155]:
# TEST
test_loss = NNmodel_relu.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1171
Test Loss: 0.12344101071357727


In [160]:
# TEST
test_loss = NNmodel_relu.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1221 
Test Loss: 0.12833793461322784


In [170]:
temp = NNmodel_relu

In [47]:
# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Instantiate the BinaryCrossentropy object
binary_crossentropy = tf.keras.losses.BinaryCrossentropy()
mean_squared_error = tf.keras.losses.MeanSquaredError()

# Calculate the binary cross-entropy on the test set
bce_loss = binary_crossentropy(true_acc, pred_acc)
mse_loss = mean_squared_error(true_dec_time, pred_dec_time)
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the binary cross-entropy loss for ACC
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss.numpy())
print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss.numpy())
print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Binary Cross-Entropy Loss on Test Set (ACC): 0.07753234
Mean Squared Error on Test Set (DEC_TIME): 0.04174434
AUC-ROC for ACC on Test Set: 0.9373275236020334


In [53]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Binary Cross-Entropy Loss on Test Set (ACC): 0.0775323442873822
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.32951998729465226
Mean Squared Error on Test Set (DEC_TIME): 0.04174433934201513
Standard Deviation of Mean Squared Error (DEC_TIME): 0.18230589016809004
AUC-ROC for ACC on Test Set: 0.9373275236020334


In [121]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Binary Cross-Entropy Loss on Test Set (ACC): 0.07135563923816815
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.32267451776033546
Mean Squared Error on Test Set (DEC_TIME): 0.04373159749057328
Standard Deviation of Mean Squared Error (DEC_TIME): 0.1352950682404268
AUC-ROC for ACC on Test Set: 0.9591503267973857


In [130]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Binary Cross-Entropy Loss on Test Set (ACC): 0.07848366322520982
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.38071743101730116
Mean Squared Error on Test Set (DEC_TIME): 0.040164007671447115
Standard Deviation of Mean Squared Error (DEC_TIME): 0.13167981521102048
AUC-ROC for ACC on Test Set: 0.9383079157588962


In [156]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Binary Cross-Entropy Loss on Test Set (ACC): 0.07200161695300075
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.3315017954138084
Mean Squared Error on Test Set (DEC_TIME): 0.0314549390599368
Standard Deviation of Mean Squared Error (DEC_TIME): 0.1142753522109199
AUC-ROC for ACC on Test Set: 0.9614015976761074


In [161]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Binary Cross-Entropy Loss on Test Set (ACC): 0.07214479029713049
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.3188554116460507
Mean Squared Error on Test Set (DEC_TIME): 0.037304191485442936
Standard Deviation of Mean Squared Error (DEC_TIME): 0.10492799258033873
AUC-ROC for ACC on Test Set: 0.956318082788671


In [168]:
import numpy as np
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# Assuming `predictions` is the output of your model on the test set (with both outputs)
predictions = NNmodel_relu.predict(X_test)

# Extract predicted ACC values (from the first column of predictions)
pred_acc = predictions[:, 0]  # This is the sigmoid output for ACC
# For DEC_TIME, extract the predicted DEC_TIME values (second column)
pred_dec_time = predictions[:, 1]  # DEC_TIME values are predicted in the second column

# Extract true ACC values from the test set
true_acc = y_test.values[:, 0]  # Assuming y_test has ACC in the first column and DEC_TIME in the second column
# Extract true DEC_TIME values from the test set (second column)
true_dec_time = y_test.values[:, 1]

# Manually calculate the binary cross-entropy for each sample
epsilon = 1e-15  # To avoid log(0)
pred_acc = np.clip(pred_acc, epsilon, 1. - epsilon)  # Ensure predictions are within a valid range
bce_loss_per_sample = - (true_acc * np.log(pred_acc) + (1 - true_acc) * np.log(1 - pred_acc))

# Manually calculate the mean squared error for each sample
mse_loss_per_sample = (true_dec_time - pred_dec_time) ** 2

# Calculate the mean and standard deviation of the losses
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

mse_loss_mean = np.mean(mse_loss_per_sample)
mse_loss_std = np.std(mse_loss_per_sample)

# Calculate AUC-ROC for ACC
auc_roc_acc = roc_auc_score(true_acc, pred_acc)

# Print the results
print("Binary Cross-Entropy Loss on Test Set (ACC):", bce_loss_mean)
print("Standard Deviation of Binary Cross-Entropy Loss (ACC):", bce_loss_std)

print("Mean Squared Error on Test Set (DEC_TIME):", mse_loss_mean)
print("Standard Deviation of Mean Squared Error (DEC_TIME):", mse_loss_std)

print("AUC-ROC for ACC on Test Set:", auc_roc_acc)

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Binary Cross-Entropy Loss on Test Set (ACC): 0.06789516959951421
Standard Deviation of Binary Cross-Entropy Loss (ACC): 0.29852265298748276
Mean Squared Error on Test Set (DEC_TIME): 0.03683127856883635
Standard Deviation of Mean Squared Error (DEC_TIME): 0.12374888387100816
AUC-ROC for ACC on Test Set: 0.9661583151779229


In [171]:
# Calculate per-sample Brier score for ACC (squared difference between predicted probabilities and actual outcomes)
brier_score_per_sample_acc = (pred_acc - true_acc) ** 2

# Calculate mean and standard deviation of Brier scores for ACC
brier_score_mean_acc = np.mean(brier_score_per_sample_acc)
brier_score_std_acc = np.std(brier_score_per_sample_acc)

# Print the mean and standard deviation of the Brier score
print("Brier Score (Mean) for ACC:", brier_score_mean_acc)
print("Brier Score (Standard Deviation) for ACC:", brier_score_std_acc)

Brier Score (Mean) for ACC: 0.019389360514702875
Brier Score (Standard Deviation) for ACC: 0.0907241613844265


In [122]:
NNmodel_relu.save('my_PatNet_models/my_PatNet_synth2.keras') 

In [157]:
NNmodel_relu.save('my_PatNet_models/my_PatNet_synth3.keras') 

In [162]:
NNmodel_relu.save('my_PatNet_models/my_PatNet_synth4.keras') 

In [169]:
NNmodel_relu.save('my_PatNet_models/my_PatNet_synth5.keras') 

In [45]:
predictions = NNmodel_relu.predict(X_test)

print(y_test[:100])
print(predictions[:100])

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
    ACC  DEC_TIME
0     0  0.198021
1     0  0.139043
2     0  0.101677
3     0  0.061641
4     0  0.375159
..  ...       ...
95    0  0.347869
96    0  0.222736
97    0  0.252203
98    0  0.159167
99    0  0.118374

[100 rows x 2 columns]
[[6.1007971e-03 2.1132132e-01]
 [4.0053842e-03 2.7607298e-01]
 [7.6191085e-03 2.0998096e-01]
 [9.9117793e-03 2.3235270e-01]
 [9.3154712e-03 2.0952725e-01]
 [6.6660219e-03 2.1536890e-01]
 [1.7548580e-02 2.1682641e-01]
 [7.6959310e-03 2.2938564e-01]
 [1.3551869e-02 2.1002644e-01]
 [6.6632475e-03 2.6524669e-01]
 [5.5844462e-03 2.4078502e-01]
 [2.8837388e-02 2.7931175e-01]
 [4.7931694e-03 2.0804340e-01]
 [1.7251724e-02 1.9636962e-01]
 [2.9598081e-02 2.6241100e-01]
 [1.7476793e-03 2.0774820e-01]
 [7.0577445e-03 2.2155711e-01]
 [1.9474514e-02 2.4155010e-01]
 [5.5687332e-01 1.8289872e+00]
 [9.8539684e-03 2.2355714e-01]
 [3.2304201e-02 2.3588505e-01]
 [8.3089471e-03 2.0820194e-01]
 [1.9236341e-03 1.9205314e-01]
 [3.041

In [23]:
predictions = NNmodel.predict(X_test)

print(y_test[:5])
print(predictions[:5])

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
   ACC  DEC_TIME
0    1  1.608411
1    0  1.099597
2    0  0.000000
3    0  0.000000
4    0  1.080268
[[0.19858105 1.196484  ]
 [0.19646715 1.2150403 ]
 [0.0229848  0.186012  ]
 [0.01423072 0.18719634]
 [0.17960978 1.2714033 ]]


### TUNING

In [140]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, LeakyReLU, Input
from keras.models import Model

def model_builder(hp):
    inputs = Input(shape=(X_train.shape[1],))
    
    # Tune the number of shared hidden layers
    hp_layers_shared = hp.Int('layers_shared', min_value=1, max_value=5, step=1)
    
    # Build shared layers
    x = inputs
    for i in range(hp_layers_shared):
        # Tune the number of units in each shared Dense layer
        hp_units_shared = hp.Int(f'units_shared_{i}', min_value=32, max_value=512, step=32)
        x = Dense(units=hp_units_shared)(x)
        
        # Tune the alpha value for LeakyReLU activation function
        hp_alpha_shared = hp.Float(f'alpha_shared_{i}', min_value=0.01, max_value=0.1, step=0.01)
        x = LeakyReLU(alpha=hp_alpha_shared)(x)
    
    # Separate paths for acc_output and dec_time_output

    # Layers for acc_output
    hp_layers_A = hp.Int('layers_acc', min_value=1, max_value=5, step=1)
    x_acc = x
    for i in range(hp_layers_A):
        # Tune the number of units in each Dense layer for acc_output
        hp_units_acc = hp.Int(f'units_acc_{i}', min_value=16, max_value=128, step=16)
        x_acc = Dense(units=hp_units_acc, activation='relu')(x_acc)
    
    # Output layer for acc_output
    acc_output = Dense(1, activation='sigmoid', name='acc_output')(x_acc)
    
    # Layers for dec_time_output
    hp_layers_T = hp.Int('layers_dec_time', min_value=1, max_value=5, step=1)
    x_dec = x
    for i in range(hp_layers_T):
        # Tune the number of units in each Dense layer for dec_time_output
        hp_units_dec_time = hp.Int(f'units_dec_time_{i}', min_value=16, max_value=128, step=16)
        x_dec = Dense(units=hp_units_dec_time, activation='relu')(x_dec)
    
    # Output layer for dec_time_output
    dec_time_output = Dense(1, activation='relu', name='dec_time_output')(x_dec)
    
    # Define the model with both outputs
    model = Model(inputs=inputs, outputs=[acc_output, dec_time_output])

    # Compile the model
    model.compile(
                  loss=custom_loss,  # Assuming you have a custom loss defined
                  metrics=['accuracy'])  # Adjust this metric as needed
    
    return model

In [135]:
pip install keras-tuner

Note: you may need to restart the kernel to use updated packages.


  Obtaining dependency information for keras-tuner from https://files.pythonhosted.org/packages/db/5d/945296512980b0827e93418514c8be9236baa6f0a1e8ca8be3a2026665b0/keras_tuner-1.4.7-py3-none-any.whl.metadata
  Obtaining dependency information for kt-legacy from https://files.pythonhosted.org/packages/16/53/aca9f36da2516db008017db85a1f3cafaee0efc5fc7a25d94c909651792f/kt_legacy-1.0.5-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/129.1 kB ? eta -:--:--
   ------ -------------------------------- 20.5/129.1 kB 330.3 kB/s eta 0:00:01
   ------------------ -------------------- 61.4/129.1 kB 825.8 kB/s eta 0:00:01
   ---------------------------------------- 129.1/129.1 kB 1.1 MB/s eta 0:00:00


In [141]:
import keras_tuner as kt
tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='my_dir',
                     project_name='intro_to_kt')

Reloading Tuner from my_dir\intro_to_kt\tuner0.json


In [137]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

In [142]:
tuner.search(X_train, y_train, epochs=50, validation_data=(X_val, y_val), callbacks=[stop_early])

# Get the optimal hyperparameters
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first densely-connected
layer is {best_hps.get('units')} and the optimal learning rate for the optimizer
is {best_hps.get('learning_rate')}.
""")


Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
4                 |4                 |layers_shared
224               |224               |units_shared_0
0.02              |0.02              |alpha_shared_0
2                 |2                 |layers_acc
16                |16                |units_acc_0
1                 |1                 |layers_dec_time
128               |128               |units_dec_time_0
0.001             |0.001             |learning_rate
320               |320               |units_shared_1
0.02              |0.02              |alpha_shared_1
224               |224               |units_shared_2
0.02              |0.02              |alpha_shared_2
128               |128               |units_dec_time_1
128               |128               |units_dec_time_2
64                |64                |units_dec_time_3
96                |96                |units_dec_time_4
32                |32                |units_shared_3
0.04             

C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
Traceback (most recent call last):
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\tuners\hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\tuners\hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 